In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

batch_size = 128
epochs = 5
latent_dim = 2

(x_train, _), (x_test, _) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_train = x_train.reshape((len(x_train), 784))

In [ ]:
encoder = tf.keras.Sequential([
    Dense(400, activation='relu'),
    Dense(latent_dim * 2)
])

decoder = tf.keras.Sequential([
    Dense(400, activation='relu'),
    Dense(784, activation='sigmoid')
])

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(batch_size)
optimizer = tf.keras.optimizers.Adam()

for epoch in range(epochs):
    train_loss = 0
    for data in dataset:
        with tf.GradientTape() as tape:
            h = encoder(data)
            mu, logvar = h[:, :latent_dim], h[:, latent_dim:]
            z = mu + tf.exp(0.5 * logvar) * tf.random.normal(shape=tf.shape(mu))
            recon_batch = decoder(z)
            
            recon_loss = -tf.reduce_sum(data * tf.math.log(recon_batch + 1e-10) + (1 - data) * tf.math.log(1 - recon_batch + 1e-10), axis=-1)
            kl_loss = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=-1)
            loss = tf.reduce_mean(recon_loss + kl_loss)
            
        grads = tape.gradient(loss, encoder.trainable_variables + decoder.trainable_variables)
        optimizer.apply_gradients(zip(grads, encoder.trainable_variables + decoder.trainable_variables))
        train_loss += loss.numpy() * len(data)
        
    print(f"Epoch {epoch+1} | Loss: {train_loss / len(x_train):.4f}")